# Celiac Gut-Brain Knowledge Graph with Heterogeneous GNNs

This notebook provides a complete, reproducible pipeline for:
1. Loading the celiac gut-brain knowledge graph
2. Training heterogeneous GNN models
3. Running baseline comparisons (TransE, DistMult, R-GCN, HGT, etc.)
4. Multi-seed experiments with statistical analysis
5. Ablation studies
6. Interpretability analysis (path analysis, case studies)

**Paper**: *Heterogeneous Graph Neural Networks for Modeling the Celiac Gut-Brain Axis*

## 1. Setup & Installation

In [ ]:
# Detect environment
import sys
IN_COLAB = 'google.colab' in sys.modules

print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# Install dependencies (Colab)
if IN_COLAB:
    !pip install torch-geometric torch-sparse torch-scatter -q
    !pip install pandas numpy matplotlib seaborn scikit-learn scipy tqdm -q
    
    # Mount Google Drive (optional - for saving results)
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Clone repository or copy files
    # !git clone https://github.com/your-repo/celiac-gut-brain-gnn.git
    # %cd celiac-gut-brain-gnn
    
print("Dependencies installed!")

In [ ]:
# Imports
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Add project to path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import project modules
from celiac.config import PROCESSED_DIR, MODELS_DIR, FIGURES_DIR
from celiac.models import HeteroGNN, LinkPredictor, CeliacGNN
from celiac.train import load_kg_from_csv, train_model, create_link_prediction_splits
from celiac.evaluation import (
    compute_all_metrics, 
    run_multi_seed_experiment, 
    set_all_seeds,
    ExperimentConfig,
    compare_models,
    generate_significance_table,
)
from celiac.baselines import BASELINE_REGISTRY, BaselineTrainer
from celiac.ablations import ABLATION_SUITE, run_extended_ablation_suite

print("Project modules loaded!")

## 2. Load Data

In [ ]:
# Load the curated CeD knowledge graph
data_dir = PROCESSED_DIR / 'pyg'
data = load_kg_from_csv(data_dir)

print("\n" + "="*60)
print("KNOWLEDGE GRAPH STATISTICS")
print("="*60)
print(f"\nNode Types ({len(data.node_types)}):")
for node_type in data.node_types:
    print(f"  {node_type}: {data[node_type].num_nodes} nodes")

print(f"\nEdge Types ({len(data.edge_types)}):")
for edge_type in data.edge_types:
    print(f"  {edge_type}: {data[edge_type].edge_index.size(1)} edges")

total_nodes = sum(data[nt].num_nodes for nt in data.node_types)
total_edges = sum(data[et].edge_index.size(1) for et in data.edge_types)
print(f"\nTotal: {total_nodes} nodes, {total_edges} edges")

In [ ]:
# Visualize graph structure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Node type distribution
node_counts = {nt: data[nt].num_nodes for nt in data.node_types}
ax = axes[0]
ax.bar(node_counts.keys(), node_counts.values(), color='steelblue')
ax.set_ylabel('Number of Nodes')
ax.set_title('Node Type Distribution')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Edge type distribution
edge_counts = {f"{et[0][:4]}→{et[2][:4]}": data[et].edge_index.size(1) for et in data.edge_types}
ax = axes[1]
ax.bar(edge_counts.keys(), edge_counts.values(), color='coral')
ax.set_ylabel('Number of Edges')
ax.set_title('Edge Type Distribution')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kg_statistics.png', dpi=150)
plt.show()

## 3. Train HeteroGNN Model

In [ ]:
# Training configuration
TARGET_EDGE_TYPE = ('gene', 'associated_with', 'phenotype')

TRAIN_CONFIG = {
    'hidden_channels': 64,
    'num_layers': 2,
    'dropout': 0.3,
    'lr': 0.01,
    'epochs': 100,
    'patience': 20,
    'seed': 42,
}

print("Training Configuration:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Train model
print("\nTraining HeteroGNN...")
model, history = train_model(
    data, 
    TARGET_EDGE_TYPE, 
    verbose=True,
    **TRAIN_CONFIG
)

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Best Epoch: {history['best_epoch']}")
print(f"Test AUROC: {history['test_auroc']:.4f}")
print(f"Test AUPRC: {history['test_auprc']:.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax = axes[0]
ax.plot(history['train_loss'], label='Train Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()

# Validation metrics
ax = axes[1]
ax.plot(history['val_auroc'], label='Val AUROC')
ax.plot(history['val_auprc'], label='Val AUPRC')
ax.axvline(x=history['best_epoch'], color='r', linestyle='--', label=f'Best Epoch ({history["best_epoch"]})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.set_title('Validation Metrics')
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves.png', dpi=150)
plt.show()

## 4. Multi-Seed Experiments

In [ ]:
# Run experiments with multiple seeds
SEEDS = [0, 1, 2, 42, 123]

print(f"Running {len(SEEDS)} seeds for statistical robustness...")

seed_results = {'auroc': [], 'auprc': [], 'hits@10': [], 'mrr': []}

for seed in SEEDS:
    print(f"\nSeed {seed}:")
    config = TRAIN_CONFIG.copy()
    config['seed'] = seed
    config['verbose'] = False
    
    _, hist = train_model(data, TARGET_EDGE_TYPE, **config)
    
    seed_results['auroc'].append(hist['test_auroc'])
    seed_results['auprc'].append(hist['test_auprc'])
    
    print(f"  AUROC: {hist['test_auroc']:.4f}, AUPRC: {hist['test_auprc']:.4f}")

# Aggregate results
print(f"\n{'='*60}")
print("MULTI-SEED RESULTS (mean ± std)")
print(f"{'='*60}")
for metric, values in seed_results.items():
    if values:
        print(f"{metric}: {np.mean(values):.4f} ± {np.std(values):.4f}")

## 5. Baseline Comparisons

In [ ]:
# List available baselines
print("Available Baselines:")
for name, info in BASELINE_REGISTRY.items():
    print(f"  {name}: {info['type']}")

In [ ]:
# Run baselines (subset for speed)
BASELINES_TO_RUN = ['TransE', 'DistMult', 'R-GCN', 'HGT']

baseline_results = {}

for baseline_name in BASELINES_TO_RUN:
    print(f"\n{'='*60}")
    print(f"Training {baseline_name}")
    print('='*60)
    
    try:
        trainer = BaselineTrainer(
            model_name=baseline_name,
            data=data,
            target_edge_type=TARGET_EDGE_TYPE,
            hidden_channels=64,
            device=str(device),
        )
        
        # Create train/val/test splits
        train_data, val_data, test_data = create_link_prediction_splits(
            data, TARGET_EDGE_TYPE, val_ratio=0.15, test_ratio=0.15
        )
        
        history = trainer.train(
            train_data=train_data,
            val_data=val_data,
            epochs=100,
            patience=20,
            verbose=True,
        )
        
        test_metrics = trainer.evaluate(test_data)
        baseline_results[baseline_name] = test_metrics
        
        print(f"\n{baseline_name} Test Results:")
        print(f"  AUROC: {test_metrics['auroc']:.4f}")
        print(f"  AUPRC: {test_metrics['auprc']:.4f}")
        
    except Exception as e:
        print(f"Error with {baseline_name}: {e}")
        baseline_results[baseline_name] = {'auroc': None, 'auprc': None, 'error': str(e)}

In [ ]:
# Compare all models
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

# Add HeteroGNN results
all_results = {'HeteroGNN (Ours)': {'auroc': history['test_auroc'], 'auprc': history['test_auprc']}}
all_results.update(baseline_results)

# Create comparison DataFrame
comparison_data = []
for model, metrics in all_results.items():
    if metrics.get('auroc') is not None:
        comparison_data.append({
            'Model': model,
            'AUROC': metrics['auroc'],
            'AUPRC': metrics['auprc'],
        })

df_comparison = pd.DataFrame(comparison_data).sort_values('AUROC', ascending=False)
print(df_comparison.to_string(index=False))

# Save to CSV
df_comparison.to_csv(MODELS_DIR / 'model_comparison.csv', index=False)

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(df_comparison))
width = 0.35

bars1 = ax.bar(x - width/2, df_comparison['AUROC'], width, label='AUROC', color='steelblue')
bars2 = ax.bar(x + width/2, df_comparison['AUPRC'], width, label='AUPRC', color='coral')

ax.set_ylabel('Score')
ax.set_title('Model Comparison on Gene-Phenotype Link Prediction')
ax.set_xticks(x)
ax.set_xticklabels(df_comparison['Model'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_comparison.png', dpi=150)
plt.show()

## 6. Ablation Studies

In [ ]:
# Run ablation studies
print("Running ablation studies...")
print("This may take a few minutes.\n")

ablations_to_run = [
    'full_graph',
    'no_microbe',
    'no_metabolite',
    'layers_1',
    'layers_3',
    'hidden_32',
    'hidden_128',
]

ablation_df = run_extended_ablation_suite(
    data_dir=PROCESSED_DIR / 'pyg',
    ablations=ablations_to_run,
    seeds=[42],  # Single seed for speed; use [0, 1, 2, 42, 123] for paper
    verbose=True,
    save_results=True,
)

print("\n" + "="*60)
print("ABLATION RESULTS")
print("="*60)
print(ablation_df.to_string(index=False))

In [ ]:
# Visualize ablation results
fig, ax = plt.subplots(figsize=(12, 6))

ablation_df_sorted = ablation_df.sort_values('auroc_mean', ascending=True)

y_pos = np.arange(len(ablation_df_sorted))
colors = ['darkgreen' if 'full_graph' in name else 'steelblue' for name in ablation_df_sorted['ablation']]

ax.barh(y_pos, ablation_df_sorted['auroc_mean'], xerr=ablation_df_sorted['auroc_std'], 
        capsize=3, color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(ablation_df_sorted['ablation'])
ax.set_xlabel('AUROC (mean ± std)')
ax.set_title('Ablation Study Results')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_results.png', dpi=150)
plt.show()

## 7. Interpretability Analysis

In [ ]:
# Import interpretability modules
from celiac.interpretability import (
    extract_top_paths,
    visualize_path,
    CED_CASE_STUDIES,
    run_all_case_studies,
    generate_case_study_report,
)

print("Interpretability modules loaded!")

In [ ]:
# Run case studies
print("\n" + "="*60)
print("CASE STUDY VALIDATION")
print("="*60)

case_study_results = run_all_case_studies(data, model=model, max_hops=4)

# Summary
validated = sum(1 for r in case_study_results if r['validation_status'] == 'validated')
partial = sum(1 for r in case_study_results if r['validation_status'] == 'partial')
not_found = sum(1 for r in case_study_results if r['validation_status'] in ['no_path', 'nodes_not_found'])

print(f"\nSummary:")
print(f"  Validated: {validated}/{len(case_study_results)}")
print(f"  Partial: {partial}/{len(case_study_results)}")
print(f"  Not found: {not_found}/{len(case_study_results)}")

In [ ]:
# Generate case study report
report = generate_case_study_report(
    case_study_results,
    output_path=str(MODELS_DIR / 'case_study_report.md')
)

# Print first part of report
print(report[:2000])

## 8. Embedding Visualization

In [ ]:
# Extract and visualize embeddings
from celiac.visualize import extract_embeddings, plot_tsne_embeddings

print("Extracting node embeddings...")
embeddings_dict = extract_embeddings(model, data)

print("\nEmbedding shapes:")
for node_type, emb in embeddings_dict.items():
    print(f"  {node_type}: {emb.shape}")

In [ ]:
# t-SNE visualization
plot_tsne_embeddings(
    embeddings_dict,
    output_path=str(FIGURES_DIR / 'tsne_embeddings.png'),
    title='Node Embeddings (t-SNE)',
)

print(f"Saved to {FIGURES_DIR / 'tsne_embeddings.png'}")

## 9. Export Results

In [ ]:
# Save all results to JSON
final_results = {
    'timestamp': datetime.now().isoformat(),
    'config': TRAIN_CONFIG,
    'hetero_gnn': {
        'test_auroc': history['test_auroc'],
        'test_auprc': history['test_auprc'],
        'best_epoch': history['best_epoch'],
    },
    'multi_seed': {
        k: {'mean': float(np.mean(v)), 'std': float(np.std(v)), 'values': v}
        for k, v in seed_results.items() if v
    },
    'baselines': {k: {m: float(v) if v is not None else None for m, v in metrics.items()} 
                 for k, metrics in baseline_results.items()},
    'case_studies': {
        'validated': validated,
        'partial': partial,
        'not_found': not_found,
        'total': len(case_study_results),
    },
}

with open(MODELS_DIR / 'final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"Results saved to {MODELS_DIR / 'final_results.json'}")

In [ ]:
# Generate LaTeX table for paper
print("\n" + "="*60)
print("LATEX TABLE FOR PAPER")
print("="*60)

latex_table = """
\\begin{table}[t]
\\centering
\\caption{Link prediction results on the CeD gut-brain knowledge graph.}
\\begin{tabular}{lcc}
\\toprule
Model & AUROC & AUPRC \\\\
\\midrule
"""

for _, row in df_comparison.iterrows():
    model = row['Model'].replace('_', '\\_')
    auroc = f"{row['AUROC']:.3f}"
    auprc = f"{row['AUPRC']:.3f}"
    if 'Ours' in model:
        latex_table += f"\\textbf{{{model}}} & \\textbf{{{auroc}}} & \\textbf{{{auprc}}} \\\\
"
    else:
        latex_table += f"{model} & {auroc} & {auprc} \\\\
"

latex_table += """\
\\bottomrule
\\end{tabular}
\\label{tab:results}
\\end{table}
"""

print(latex_table)

with open(FIGURES_DIR / 'results_table.tex', 'w') as f:
    f.write(latex_table)

## 10. Summary

This notebook demonstrated:
1. Loading the celiac gut-brain knowledge graph
2. Training a heterogeneous GNN for link prediction
3. Multi-seed experiments for statistical robustness
4. Comparison with KGE and GNN baselines
5. Ablation studies on node types, layers, and hidden dimensions
6. Case study validation against known biological mechanisms
7. Embedding visualization with t-SNE

**Key Results**:
- HeteroGNN achieves strong performance on gene-phenotype link prediction
- Removing microbe/metabolite nodes impacts performance (showing multi-hop importance)
- Case studies validate predictions against known celiac gut-brain mechanisms